# 🎯 Lab 01: 中文情感分析 (Sentiment Analysis)

**使用 TF-IDF + 機器學習方法對中文評論進行情感二分類**

---

### 🎯 學習目標
- 掌握中文文本預處理 (jieba 分詞、停用詞過濾)
- 理解 TF-IDF 特徵工程
- 比較 LogReg / SVM / Random Forest
- 視覺化混淆矩陣和模型對比

### 📊 數據集
**ChnSentiCorp** (~9,600 條攜程酒店評論)

### 🛠 技術棧
Scikit-learn, jieba, Pandas, Matplotlib

### ⏱ 預計時間
- GPU: 不需要
- 運行時間: ~3 分鐘

---

## 1. 環境準備

In [ ]:
# 安裝依賴 (Colab 自動重啟時需要)
!pip install -q scikit-learn jieba pandas matplotlib datasets
print('✅ 環境準備完成')

## 2. 載入數據

In [ ]:
import os, re, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import jieba

# 設定中文字型 (Colab 環境)
plt.rcParams['font.sans-serif'] = ['Noto Sans CJK SC', 'PingFang TC', 'SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

print('✅ 套件載入完成')

In [ ]:
def load_sentiment_data():
    """下載 ChnSentiCorp，失敗則用本地示例"""
    try:
        print('📥 嘗試從 HuggingFace 下載 ChnSentiCorp...')
        from datasets import load_dataset
        ds_train = load_dataset('lansinuote/ChnSentiCorp', split='train')
        ds_test = load_dataset('lansinuote/ChnSentiCorp', split='test')
        train_df = pd.DataFrame({'text': ds_train['text'], 'label': ds_train['label']})
        test_df = pd.DataFrame({'text': ds_test['text'], 'label': ds_test['label']})
        print(f'✅ 訓練: {len(train_df)}, 測試: {len(test_df)}')
    except Exception as e:
        print(f'⚠️  在線下載失敗: {e}')
        print('   使用本地示例數據...')
        data_dir = './data' if os.path.exists('./data') else '/content/data'
        if not os.path.exists(data_dir + '/sample_train.csv'):
            os.makedirs(data_dir, exist_ok=True)
            !wget -q https://raw.githubusercontent.com/chanxaviersy/nlp-ml-knowledge-base/main/docs/08-labs/01-sentiment-analysis/data/sample_train.csv -O {data_dir}/sample_train.csv
            !wget -q https://raw.githubusercontent.com/chanxaviersy/nlp-ml-knowledge-base/main/docs/08-labs/01-sentiment-analysis/data/sample_test.csv -O {data_dir}/sample_test.csv
        train_df = pd.read_csv(f'{data_dir}/sample_train.csv')
        test_df = pd.read_csv(f'{data_dir}/sample_test.csv')
    train_df = train_df.dropna().reset_index(drop=True)
    test_df = test_df.dropna().reset_index(drop=True)
    return train_df, test_df

train_df, test_df = load_sentiment_data()
print(f'\n📊 訓練集: {len(train_df)}')
print(f'📊 測試集: {len(test_df)}')
print(f'\n類別分佈:\n{train_df["label"].value_counts()}')
train_df.head()

## 3. 文本預處理

In [ ]:
def preprocess_chinese(text):
    text = re.sub(r'http\S+', '', str(text))
    text = re.sub(r'[^\u4e00-\u9fffA-Za-z0-9 ]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    words = jieba.lcut(text)
    return ' '.join(w for w in words if len(w) > 1)

print('🔧 預處理文本...')
X_train = [preprocess_chinese(t) for t in train_df['text']]
y_train = train_df['label'].values
X_test = [preprocess_chinese(t) for t in test_df['text']]
y_test = test_df['label'].values

print(f'\n示例 (原始): {train_df["text"].iloc[0]}')
print(f'示例 (處理): {X_train[0]}')

## 4. 訓練多個模型

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score

def build_pipeline(model_name='logreg'):
    tfidf = TfidfVectorizer(max_features=50000, ngram_range=(1, 2), min_df=2, sublinear_tf=True)
    if model_name == 'logreg':
        clf = LogisticRegression(C=1.0, max_iter=1000, random_state=42)
    elif model_name == 'svm':
        clf = LinearSVC(C=1.0, max_iter=2000, random_state=42)
    elif model_name == 'rf':
        clf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
    return Pipeline([('tfidf', tfidf), ('clf', clf)])

results = {}
preds_dict = {}
for name in ['logreg', 'svm', 'rf']:
    print(f'\n🤖 訓練 {name}...')
    pipe = build_pipeline(name)
    pipe.fit(X_train, y_train)
    preds = pipe.predict(X_test)
    preds_dict[name] = preds
    results[name] = {
        'accuracy': accuracy_score(y_test, preds),
        'f1': f1_score(y_test, preds, average='binary'),
    }
    print(f'  accuracy={results[name]["accuracy"]:.4f}, f1={results[name]["f1"]:.4f}')

print('\n📈 模型對比:')
pd.DataFrame(results).T

## 5. 視覺化

In [ ]:
import seaborn as sns

# 模型對比柱狀圖
fig, ax = plt.subplots(figsize=(8, 5))
metrics_df = pd.DataFrame(results).T
metrics_df.plot(kind='bar', ax=ax)
ax.set_title('Model Performance Comparison')
ax.set_ylabel('Score')
ax.set_ylim([0, 1.1])
ax.legend(loc='lower right')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

# 最佳模型的混淆矩陣
best_name = max(results, key=lambda k: results[k]['f1'])
cm = confusion_matrix(y_test, preds_dict[best_name])
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['負面','正面'], yticklabels=['負面','正面'])
ax.set_title(f'Confusion Matrix - {best_name}')
ax.set_xlabel('預測')
ax.set_ylabel('真實')
plt.tight_layout()
plt.show()

## 6. 互動測試

修改下面的文本，測試自己的評論！

In [ ]:
# 重新訓練最好的模型 (LogReg 通常最佳)
best_pipe = build_pipeline('logreg')
best_pipe.fit(X_train, y_train)

def predict_sentiment(text):
    processed = preprocess_chinese(text)
    pred = best_pipe.predict([processed])[0]
    proba = best_pipe.predict_proba([processed])[0]
    label = '😊 正面' if pred == 1 else '😞 負面'
    confidence = proba[pred]
    print(f'文本: {text}')
    print(f'預測: {label} (置信度: {confidence:.2%})')
    return pred

# 測試樣本
test_texts = [
    '這家餐廳的菜非常好吃，服務也很周到',
    '完全不符合描述，質量差，物流慢',
    '出乎意料的驚喜，強烈推薦！',
    '非常失望，浪費金錢和時間',
]
for t in test_texts:
    predict_sentiment(t)
    print('---')

## 7. 總結

### 🎉 恭喜完成 Lab 01！

**你學到了**：
- ✅ 中文預處理 (jieba 分詞)
- ✅ TF-IDF 特徵工程
- ✅ 3 個分類模型對比
- ✅ 評估指標 (accuracy, F1, confusion matrix)

### 🚀 下一步

👉 [Lab 02: CNN 文本分類](../02-text-classification-cnn/lab.ipynb) — 使用 PyTorch 從零實現 TextCNN

### 📚 延伸閱讀

- [BERT 中文情感分析 (Lab 03)](../03-transformer-sentiment/lab.ipynb)
- [TF-IDF 原理](https://en.wikipedia.org/wiki/Tf%E2%80%93idf)
- [sklearn 文本分類](https://scikit-learn.org/stable/tutorial/text_analytics/working_with_text_data.html)